# Computer Vision – Convolutions and Image Classification (PyTorch)

In this tutorial we will:
1. Load CIFAR-10
2. Build a simple baseline classifier using Flatten + KNN
3. Build a simple Convolutional Neural Network (CNN)
4. Compare their performance
5. Define your assignment


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


device(type='cpu')

## Load CIFAR-10


In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)


Files already downloaded and verified
Files already downloaded and verified


## Baseline: Flatten + KNN


In [ ]:
subset_size = 5000
X_train, y_train = [], []

for i in range(subset_size):
    img, label = trainset[i]
    X_train.append(img.numpy().flatten())
    y_train.append(label)

X_test, y_test = [], []
for i in range(1000):
    img, label = testset[i]
    X_test.append(img.numpy().flatten())
    y_test.append(label)

X_train = np.array(X_train) # Convert to numpy array, which is required by scikit-learn classifiers
X_test = np.array(X_test)

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
print('KNN Accuracy:', accuracy_score(y_test, y_pred))


/Users/eugenio/Documents/Computer_Vision/.venv/lib/python3.11/site-packages/threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


KNN Accuracy: 0.261


## Simple CNN


In [4]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32*8*8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [5]:
epochs = 5
for epoch in range(epochs):
    running_loss = 0.0
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss/len(trainloader):.4f}')


Epoch 1, Loss: 1.5327
Epoch 2, Loss: 1.1926
Epoch 3, Loss: 1.0399
Epoch 4, Loss: 0.9364
Epoch 5, Loss: 0.8590


# Assignment

1. Choose a different dataset (MNIST, FashionMNIST, etc.)
2. Implement a baseline and a CNN
3. Modify at least one architectural component
4. Compare accuracy
5. Present in 5 minutes explaining results


# Rubric (10 points)

- Baseline implemented (2)
- CNN implemented (2)
- Architectural modification (2)
- Clear comparison (2)
- Conceptual explanation (2)
